In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_frontraise_feedback(landmarks):
    # Left Arm: Shoulder Angle for height (Hip-Shoulder-Elbow)
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]

    # Right Arm coordinates for symmetry check
    right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
    right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]


    # Calculate angles
    left_shoulder_angle = calculate_angle(left_hip, left_shoulder, left_elbow)
    right_shoulder_angle = calculate_angle(right_hip, right_shoulder, right_elbow)
    avg_shoulder_angle = (left_shoulder_angle + right_shoulder_angle) / 2

    # Target Angle for height is 90 degrees relative to the hip/body line

    # --- Feedback Logic ---
    if avg_shoulder_angle < 75:
        feedback = "Too Low! Raise Arms Higher"
    elif 75 <= avg_shoulder_angle <= 105:
        feedback = "Perfect Height!"
    else:
        feedback = "Too High! Stop at Shoulder Height"

    # --- Accuracy Calculation ---
    # Score based on height (Target 90)
    accuracy = max(0, 100 - abs(avg_shoulder_angle - 90) * 2.5)
    accuracy = max(0, min(100, int(accuracy)))

    return feedback, int(accuracy)

In [ ]:
def analyze_frontraise(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error: Could not open input video."

    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    # Robust Codec Fix: Use 'MJPG' and .avi extension
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    output_video = "output_frontraise.avi"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    if not out.isOpened():
        return "Error: Could not create output video writer."

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Check visibility of a critical landmark before calculation
            if landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].visibility > 0.6:
                mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

                feedback, accuracy = check_frontraise_feedback(landmarks)

                color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

                cv2.putText(image, feedback, (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
                cv2.putText(image, f"Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

            else:
                 cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

        else:
             cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)


        out.write(image)

    cap.release()
    out.release()
    return output_video

# Gradio Interface
gr.Interface(
    fn=analyze_frontraise,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Front Raise Form Analyzer",
    description="Upload a video of your front raises, and get feedback on your height!",
).launch(share=True, debug=True)